# Fase 1 — Fine-Tuning VGG16 (classificação binária)

**Classes:** `class_0` = Floresta | `class_1` = Oceano

**Origem do dataset:** imagens sintéticas geradas localmente com `scripts/generate_dataset.py` (Python + Pillow). 100 imagens no total (50 por classe), com variação de céu, iluminação, quantidade e posição dos elementos. Não foi utilizado download de dataset pronto de plataformas online.

## 1. Configuração e imports

In [ ]:
import os
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from tensorflow.keras import layers, models
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, TensorBoard
from tensorflow.keras.preprocessing.image import ImageDataGenerator

print(f"TensorFlow {tf.__version__}")
print(f"GPU disponível: {len(tf.config.list_physical_devices('GPU')) > 0}")

## 2. Caminhos e hiperparâmetros

In [ ]:
# Ajuste se o notebook estiver em notebooks/ e data/ na raiz do projeto
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
LOG_DIR = PROJECT_ROOT / "logs" / "fit" / datetime.now().strftime("%Y%m%d-%H%M%S")
MODEL_PATH = PROJECT_ROOT / "models" / "vgg16_finetuned.keras"

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 8
SEED = 42
VAL_SPLIT = 0.2  # 20% para validação (10 imagens por classe)
EPOCHS_HEAD = 8
EPOCHS_FINETUNE = 12

MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

assert (DATA_DIR / "class_0").exists() and (DATA_DIR / "class_1").exists(), (
    f"Execute scripts/generate_dataset.py antes. Pasta esperada: {DATA_DIR}")

print(f"DATA_DIR: {DATA_DIR}")
print(f"LOG_DIR (TensorBoard): {LOG_DIR}")

## 3. Pré-processamento e divisão treino/validação

In [ ]:
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=VAL_SPLIT,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
)

val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=VAL_SPLIT,
)

train_generator = train_datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    subset="training",
    seed=SEED,
    shuffle=True,
)

val_generator = val_datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    subset="validation",
    seed=SEED,
    shuffle=False,
)

class_names = list(train_generator.class_indices.keys())
print("Classes:", train_generator.class_indices)
print(f"Amostras treino: {train_generator.samples}")
print(f"Amostras validação: {val_generator.samples}")

## 4. Modelo VGG16 + cabeça classificadora

In [ ]:
def build_model(trainable_backbone: bool = False) -> tf.keras.Model:
    base_model = VGG16(include_top=False, weights="imagenet", input_shape=(*IMAGE_SIZE, 3))
    base_model.trainable = trainable_backbone

    inputs = layers.Input(shape=(*IMAGE_SIZE, 3))
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)

    model = models.Model(inputs, outputs)
    model.base_model = base_model
    return model

model = build_model(trainable_backbone=False)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
    ],
)
model.summary()

## 5. TensorBoard e treino (fase 1 — só a cabeça)

In [ ]:
tensorboard_callback = TensorBoard(
    log_dir=str(LOG_DIR),
    histogram_freq=0,
    write_graph=False,
)

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True,
)

checkpoint = ModelCheckpoint(
    filepath=str(MODEL_PATH),
    monitor="val_loss",
    save_best_only=True,
    verbose=1,
)

history_head = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS_HEAD,
    callbacks=[tensorboard_callback, early_stopping, checkpoint],
)

print("\nPara ver TensorBoard no terminal:")
print(f"  tensorboard --logdir {LOG_DIR.parent}")

## 6. Fine-Tuning (fase 2 — liberar camadas finais da VGG16)

In [ ]:
base_model = model.base_model
base_model.trainable = True

# Congela camadas iniciais; ajusta apenas as últimas blocos convolucionais
for layer in base_model.layers[:-4]:
    layer.trainable = False

trainable_count = sum(1 for layer in base_model.layers if layer.trainable)
print(f"Camadas treináveis no backbone: {trainable_count}")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
    ],
)

history_finetune = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS_FINETUNE,
    callbacks=[tensorboard_callback, early_stopping, checkpoint],
)

## 7. Curvas de treino (loss e acurácia)

In [ ]:
def merge_histories(h1, h2):
    merged = {}
    for key in h1.history:
        merged[key] = h1.history[key] + h2.history[key]
    return merged

history = merge_histories(history_head, history_finetune)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history["loss"], label="treino")
axes[0].plot(history["val_loss"], label="validação")
axes[0].set_title("Loss")
axes[0].set_xlabel("Época")
axes[0].legend()

axes[1].plot(history["accuracy"], label="treino")
axes[1].plot(history["val_accuracy"], label="validação")
axes[1].set_title("Acurácia")
axes[1].set_xlabel("Época")
axes[1].legend()

plt.tight_layout()
plt.show()

## 8. Métricas finais na validação

In [ ]:
val_generator.reset()
y_prob = model.predict(val_generator, verbose=0).ravel()
y_true = val_generator.classes
y_pred = (y_prob >= 0.5).astype(int)

metrics = {
    "loss": float(model.evaluate(val_generator, verbose=0)[0]),
    "accuracy": accuracy_score(y_true, y_pred),
    "precision": precision_score(y_true, y_pred, zero_division=0),
    "recall": recall_score(y_true, y_pred, zero_division=0),
    "f1": f1_score(y_true, y_pred, zero_division=0),
}

print("=== Métricas (validação) ===")
for name, value in metrics.items():
    print(f"{name:12s}: {value:.4f}")

print("\n=== Relatório por classe ===")
print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

## 9. Matriz de confusão

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(5, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
)
plt.xlabel("Predito")
plt.ylabel("Real")
plt.title("Matriz de confusão (validação)")
plt.tight_layout()
plt.show()

## 10. Discussão dos resultados

**Interpretação (edite após rodar com seus números):**

- Com imagens sintéticas bem separadas (floresta vs oceano), é esperado **acurácia alta** (>90%) na validação, pois as classes são visualmente distintas.
- Se a acurácia de treino é muito maior que a de validação, pode indicar **overfitting** — com apenas 80 imagens de treino isso é comum; aumentar dados ou augmentation ajuda.
- **Precision** alta com **recall** baixo indica que o modelo é conservador (erra menos falsos positivos, mas deixa passar alguns casos).
- A matriz de confusão mostra se o erro está concentrado em uma classe (ex.: confundir oceano com floresta).

**Conclusão:** para este dataset sintético, resultados com acurácia e F1 acima de 0,90 costumam ser **bons** e validam que o pipeline de fine-tuning está correto. Em um dataset real (fotos com ruído), valores entre 0,70–0,85 seriam mais típicos e ainda aceitáveis para 100 imagens.

**TensorBoard:** loss e acurácia (treino e validação) foram registradas em `logs/fit/`. Use `tensorboard --logdir logs/fit` para visualizar.